# LAB 4

## Author: Enrique Posada

**Create an original code that uses the following speech tasks:**

- Local Speech to text
- Local Text to speech
- External API Speech to text
- External API Tex to speech
- Local or external API speech task beyond STT and TTS

In [36]:
import whisper
import os
from gtts import gTTS
from transformers import pipeline
import pyttsx3
from huggingface_hub import InferenceClient
import assemblyai as aai
import requests
import speech_recognition as sr
import pygame
import numpy as np
import io
import tempfile
import soundfile as sf

## Local Speech to Text

In [2]:
# Load model
local_stt = whisper.load_model("base")

# Select audio
audio_file = "m0005_us_m0005_00302.wav"

# Transcribe
local_transcription = local_stt.transcribe(audio_file, task="transcribe")

d:\Kike\Programacion\Language\lang\Lib\site-packages\whisper\transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


In [3]:
# Print result
print("\n=== TRANSCRIPTION SUMMARY ===")
print(f"Language: {local_transcription['language']}")
print(f"Full Text: {local_transcription['text'].strip()}")


=== TRANSCRIPTION SUMMARY ===
Language: en
Full Text: Take the Jewish idea of forgiveness.


## Local Text to Speech

In [4]:
# Local STT
local_text = "I need to finish this lab so I can work on my project"
engine = pyttsx3.init()

# Save to file
engine.save_to_file(local_text, "output_local_tts.wav")

# Run the queue
engine.runAndWait()

print("Audio saved as output_local_tts.wav")

Audio saved as output_local_tts.wav


## External API Speech to Text

In [16]:
# Set up inference client
aai.settings.api_key = "817fd5bdbdcd43e6a74c312ca16f9bf6"

config = aai.TranscriptionConfig(
    speech_models=["universal-2"]
)

transcriber = aai.Transcriber()
result = transcriber.transcribe(audio_file, config=config)

print(result.text)

Take the Jewish idea of forgiveness.


## External Text to Speech

In [12]:
# External model
external_text = "I need to finish this lab so I can work on my project"
tts = gTTS(text=external_text, lang="en")


output_file = "output_external_tts.mp3"
tts.save(output_file)

print("Audio saved as output_external_tts.wav")

Audio saved as output_external_tts.wav


## Local speech task beyond STT and TTS - Chatbot

In [37]:
# Load whisper model 
model = whisper.load_model("base")

# STT
def listen():
    r = sr.Recognizer()
    r.energy_threshold = 4000
    r.pause_threshold = 1.0 
    r.dynamic_energy_threshold = False  
    with sr.Microphone() as source:
        print("Listening...")
        audio = r.listen(source, timeout=5, phrase_time_limit=15)

    with tempfile.NamedTemporaryFile(suffix=".wav", delete=False) as f:
        f.write(audio.get_wav_data())
        temp_path = f.name

    result = model.transcribe(temp_path, fp16=False)
    return result["text"]

# LLM
def chat(text, history=[]):
    if not history:
        history.append({
            "role": "system",
            "content": """You are a helpful voice assistant called Jarvis.
            
Your instructions:
- Respond in the same language the user speaks
- Keep answers short and concise since they will be spoken out loud
- Avoid using bullet points, markdown, special characters or emojis
- Be friendly and conversational
- If you don't know something, say so clearly"""
        })
    
    history.append({"role": "user", "content": text})
    
    r = requests.post(
        "http://localhost:11434/api/chat",
        json={
            "model": "gemma3:1b",
            "messages": history,
            "stream": False
        }
    )
    
    reply = r.json()["message"]["content"].strip()
    history.append({"role": "assistant", "content": reply})
    return reply, history

# TTS
def speak(text, lang="es"):
    tts = gTTS(text=text, lang=lang)
    
    # save to memory 
    mp3_fp = io.BytesIO()
    tts.write_to_fp(mp3_fp)
    mp3_fp.seek(0)
    
    pygame.mixer.init()
    pygame.mixer.music.load(mp3_fp, "mp3")
    pygame.mixer.music.play()
    while pygame.mixer.music.get_busy():
        pygame.time.Clock().tick(10)

# Main loop
history = []
while True:
    user_input = listen()
    print(f"You: {user_input}")
    
    response, history = chat(user_input, history)
    print(f"Bot: {response}")
    
    speak(response)

Listening...
You:  Hello, beautiful. Do you hear me?
Bot: Yes, I do. Hello there!
Listening...


WaitTimeoutError: listening timed out while waiting for phrase to start

## Link of Video